## Phase 1 — Generate Answers and Score Claims (Kaggle GPU)

### Cell 1 — Setup

In [ ]:
!pip -q install transformers bitsandbytes accelerate
import sys; sys.path.insert(0, "/kaggle/working/severity-aware-conformal")  # repo path
from sac.kqa_loader import load_kqa, gold_statements
from sac.hf_backend import HFBackend
from sac.decompose import decompose
from sac.scoring import score_claim
from sac.cache import append_claims, existing_claim_ids
from sac.crc import Claim

### Cell 2 — Load model + data (N_QUESTIONS = 50)

In [ ]:
backend = HFBackend()
items = load_kqa("/kaggle/input/kqa/questions.jsonl")[:50]   # confirm path/field names first
CACHE = "/kaggle/working/claims_phase1.jsonl"
done = existing_claim_ids(CACHE)

### Cell 3 — Checkpointed generation loop (resumes via `done`)

In [ ]:
for it in items:
    if any(cid.startswith(it.qid + "_") for cid in done):
        continue                                  # this question already cached
    answer = backend.generate(f"Question: {it.question}\nAnswer:")
    claims_text = decompose(it.question, answer, backend)
    rows = []
    for j, ct in enumerate(claims_text):
        conf = score_claim(ct, backend)
        rows.append(Claim(text=ct, confidence=conf,
                          answer_id=it.qid, claim_id=f"{it.qid}_c{j}"))
    append_claims(CACHE, rows)                     # checkpoint after every question
    print(it.qid, "->", len(rows), "claims")

### Cell 4 — Persist to a Kaggle Dataset / Drive so the GPU pass is never repeated

In [ ]:
from sac.cache import load_claims
print("total cached claims:", len(load_claims(CACHE)))
# Then: "Save Version" with the output, or copy CACHE to a Kaggle Dataset.